# Spider Error Analysis

Notebook for analyzing the current best full Spider dev run:
- global metrics and cost
- top problematic databases
- natural-language questions from problematic databases
- clearly wrong predictions outside those databases

Current target run:
`outputs/ablation_cheap_gen_more_candidates_20260407_204301_20260407_204302.json`

In [1]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path
from pprint import pprint

RUN_PATH = Path('../outputs/ablation_cheap_gen_more_candidates_20260407_204301_20260407_204302.json')
assert RUN_PATH.exists(), RUN_PATH

payload = json.loads(RUN_PATH.read_text())
predictions = payload['predictions']
metrics = payload['metrics']
timings = payload['timings']
cost = payload['cost']

print('Run file:', RUN_PATH)
print('EX:', round(metrics['execution_accuracy'] * 100, 2), '%')
print('EM:', round(metrics['exact_match'] * 100, 2), '%')
print('Errors:', metrics['errors'])
print('Avg time/example:', timings['avg_time_per_example_s'])
print('Total cost USD:', round(cost['total_cost_usd'], 4))
print('Avg cost/example USD:', round(cost['avg_cost_per_example_usd'], 5))
print('Total predictions:', len(predictions))

Run file: ../outputs/ablation_cheap_gen_more_candidates_20260407_204301_20260407_204302.json
EX: 72.34 %
EM: 33.66 %
Errors: 28
Avg time/example: 3.5758
Total cost USD: 21.9847
Avg cost/example USD: 0.02126
Total predictions: 1034


In [2]:
# Top problematic databases by hard errors
error_db = Counter()
for row in predictions:
    if row.get('error_message'):
        error_db[row['db_id']] += 1

top_problem_dbs = [db for db, _ in error_db.most_common(8)]
print('Top problematic DBs:')
for db, n in error_db.most_common(12):
    print(f'{db:30s} {n}')

top_problem_dbs

Top problematic DBs:
dog_kennels                    9
car_1                          5
student_transcripts_tracking   4
concert_singer                 2
flight_2                       2
employee_hire_evaluation       2
wta_1                          2
world_1                        2


['dog_kennels',
 'car_1',
 'student_transcripts_tracking',
 'concert_singer',
 'flight_2',
 'employee_hire_evaluation',
 'wta_1',
 'world_1']

In [3]:
# Natural-language questions from the main problematic DBs
focus_dbs = ['dog_kennels', 'car_1', 'student_transcripts_tracking']
focus_questions = {db: [] for db in focus_dbs}

for row in predictions:
    db = row['db_id']
    if db in focus_questions and row['question'] not in focus_questions[db]:
        focus_questions[db].append(row['question'])

for db in focus_dbs:
    print('\n' + '=' * 80)
    print(db)
    print('=' * 80)
    for i, question in enumerate(focus_questions[db], 1):
        print(f'{i:2d}. {question}')


dog_kennels
 1. Which states have both owners and professionals living there?
 2. Find the states where both owners and professionals live.
 3. What is the average age of the dogs who have gone through any treatments?
 4. Find the average age of the dogs who went through treatments.
 5. Which professionals live in the state of Indiana or have done treatment on more than 2 treatments? List his or her id, last name and cell phone.
 6. Find the id, last name and cell phone of the professionals who live in the state of Indiana or have performed more than two treatments.
 7. Which dogs have not cost their owner more than 1000 for treatment? List the dog names.
 8. What are the names of the dogs for which the owner spent more than 1000 for treatment?
 9. Which first names are used for professionals or owners but are not used as dog names?
10. Find the first names that are used for professionals or owners but are not used as dog names.
11. Which professional did not operate any treatment on 

In [9]:
# dog_kennels: the 9 hard-error rows in full detail

dog_rows = [
    row for row in predictions
    if row['db_id'] == 'dog_kennels' and row.get('error_message')
]

print('dog_kennels hard-error rows:', len(dog_rows))
for i, row in enumerate(dog_rows, 1):
    print('\n' + '=' * 120)
    print(f'[{i}] QUESTION: {row["question"]}')
    print(f'PREDICTED SQL: {row["predicted_sql"]}')
    print(f'GOLD SQL:      {row["gold_sql"]}')
    print(f'EXEC MATCH:    {row["execution_match"]}')
    print(f'ERROR:         {row.get("error_message")}')
    print('WARNINGS:')
    for w in row.get('warnings', []):
        print('  -', w)


dog_kennels hard-error rows: 9

[1] QUESTION: What are the names of the dogs for which the owner spent more than 1000 for treatment?
PREDICTED SQL: SELECT name FROM Dogs WHERE dog_id IN (SELECT dog_id FROM Treatments GROUP BY dog_id HAVING SUM(cost) > 1000);
GOLD SQL:      SELECT name FROM Dogs WHERE dog_id NOT IN( SELECT dog_id FROM Treatments GROUP BY dog_id HAVING sum(cost_of_treatment)  >  1000 )
EXEC MATCH:    False
ERROR:         (sqlite3.OperationalError) no such column: amount
[SQL: SELECT name FROM Dogs WHERE dog_id IN (SELECT dog_id FROM Treatments GROUP BY dog_id HAVING SUM(amount) > 1000);]
(Background on this error at: https://sqlalche.me/e/20/e3q8)
WARNINGS:
  - query_sketcher: failed to parse JSON response
  - query_sketcher: repaired malformed sketch with structured-output fallback
  - generator: using reduced ensemble budget for moderate complexity
  - execution_filter: (sqlite3.OperationalError) no such column: T2.cost
[SQL: SELECT T1.name FROM Dogs AS T1 JOIN Treatme

In [11]:
# dog_kennels: compact table view for the 9 hard-error rows

import pandas as pd

pd.set_option('display.max_colwidth', 200)

dog_table = pd.DataFrame([
    {
        'question': row['question'],
        'predicted_sql': row['predicted_sql'],
        'gold_sql': row['gold_sql'],
        'error_message': row.get('error_message'),
    }
    for row in predictions
    if row['db_id'] == 'dog_kennels' and row.get('error_message')
])

dog_table

,question,predicted_sql,gold_sql,error_message
0,What are the names of the dogs for which the owner spent more than 1000 for treatment?,SELECT name FROM Dogs WHERE dog_id IN (SELECT dog_id FROM Treatments GROUP BY dog_id HAVING SUM(cost) > 1000);,SELECT name FROM Dogs WHERE dog_id NOT IN( SELECT dog_id FROM Treatments GROUP BY dog_id HAVING sum(cost_of_treatment) > 1000 ),(sqlite3.OperationalError) no such column: amount\n[SQL: SELECT name FROM Dogs WHERE dog_id IN (SELECT dog_id FROM Treatments GROUP BY dog_id HAVING SUM(amount) > 1000);]\n(Background on this erro...
1,Tell me the owner id and last name of the owner who spent the most on treatments of his or her dogs.,"SELECT O.owner_id, O.last_name FROM Owners O JOIN Dogs D ON O.owner_id = D.owner_id JOIN Treatments T ON D.dog_id = T.dog_id GROUP BY O.owner_id ORDER BY SUM(T.cost) DESC LIMIT 1;","SELECT T1.owner_id , T1.last_name FROM Owners AS T1 JOIN Dogs AS T2 ON T1.owner_id = T2.owner_id JOIN Treatments AS T3 ON T2.dog_id = T3.dog_id GROUP BY T1.owner_id ORDER BY count(*) DESC LIM...",schema_validation: unknown column 'T.treatment_cost'
2,Give me the description of the treatment type whose total cost is the lowest.,SELECT T2.treatment_type_description FROM Treatments AS T1 JOIN Treatment_Types AS T2 ON T1.treatment_type_code = T2.treatment_type_code GROUP BY T2.treatment_type_description ORDER BY SUM(T1.trea...,SELECT T1.treatment_type_description FROM Treatment_types AS T1 JOIN Treatments AS T2 ON T1.treatment_type_code = T2.treatment_type_code GROUP BY T1.treatment_type_code ORDER BY sum(cost_of_trea...,schema_validation: unknown column 'T1.treatment_cost'
3,Find the id and cell phone of the professionals who operate two or more types of treatments.,"SELECT p.professional_id, p.cell_number FROM Professionals p JOIN Treatments t ON p.professional_id = t.professional_id GROUP BY p.professional_id, p.cell_number HAVING COUNT(DISTINCT t.treatment_...","SELECT T1.professional_id , T1.cell_number FROM Professionals AS T1 JOIN Treatments AS T2 ON T1.professional_id = T2.professional_id GROUP BY T1.professional_id HAVING count(*) >= 2",schema_validation: unknown column 't.treatment_code'
4,Which professionals have operated a treatment that costs less than the average? Give me theor first names and last names.,"SELECT DISTINCT P.first_name, P.last_name FROM Professionals P JOIN Treatments T ON P.professional_id = T.professional_id WHERE T.cost < (SELECT AVG(cost) FROM Treatments);","SELECT DISTINCT T1.first_name , T1.last_name FROM Professionals AS T1 JOIN Treatments AS T2 WHERE cost_of_treatment < ( SELECT avg(cost_of_treatment) FROM Treatments )",schema_validation: unknown column 'T.cost'
5,What are the cost and treatment type description of each treatment?,"SELECT T1.treatment_type_code, T2.treatment_type_description FROM Treatments AS T1 JOIN Treatment_Types AS T2 ON T1.treatment_type_code = T2.treatment_type_code;","SELECT T1.cost_of_treatment , T2.treatment_type_description FROM Treatments AS T1 JOIN treatment_types AS T2 ON T1.treatment_type_code = T2.treatment_type_code",schema_validation: unknown column 'T1.treatment_cost'
6,Show me the cost of the most recently performed treatment.,SELECT treatment_cost FROM Treatments ORDER BY treatment_date DESC LIMIT 1;,SELECT cost_of_treatment FROM Treatments ORDER BY date_of_treatment DESC LIMIT 1,(sqlite3.OperationalError) no such column: cost\n[SQL: SELECT cost FROM Treatments ORDER BY treatment_date DESC LIMIT 1;]\n(Background on this error at: https://sqlalche.me/e/20/e3q8)
7,List the first name of all the professionals along with the description of the treatment they have done.,"SELECT Professionals.first_name, Treatments.description FROM Professionals JOIN Treatments ON Professionals.professional_id = Treatments.professional_id;","SELECT DISTINCT T1.first_name , T3.treatment_type_description FROM professionals AS T1 JOIN Treatments AS T2 ON T1.professional_id = T2.professional_id JOIN Treatment_types AS T3 ON T2.treatmen...

In [12]:
# car_1: compact table view for hard-error rows

import pandas as pd

pd.set_option('display.max_colwidth', 200)

car_table = pd.DataFrame([
    {
        'question': row['question'],
        'predicted_sql': row['predicted_sql'],
        'gold_sql': row['gold_sql'],
        'error_message': row.get('error_message'),
    }
    for row in predictions
    if row['db_id'] == 'car_1' and row.get('error_message')
])

car_table

,question,predicted_sql,gold_sql,error_message
0,Find the model of the car whose weight is below the average weight.,SELECT model FROM cars_data WHERE weight < (SELECT AVG(weight) FROM cars_data);,SELECT T1.model FROM CAR_NAMES AS T1 JOIN CARS_DATA AS T2 ON T1.MakeId = T2.Id WHERE T2.Weight < (SELECT avg(Weight) FROM CARS_DATA),(sqlite3.OperationalError) no such column: model\n[SQL: SELECT model FROM car_makers WHERE weight < (SELECT AVG(weight) FROM car_makers);]\n(Background on this error at: https://sqlalche.me/e/20/e...
1,What is the largest amount of horsepower for the models with 3 cylinders and what make is it?,SELECT Maker FROM car_makers WHERE Id IN ( SELECT Id FROM car_makers WHERE Cylinders = 3 AND Horsepower = ( SELECT MAX(Horsepower) FROM car_makers WHERE Cylinders = 3 ) );,"SELECT T2.horsepower , T1.Make FROM CAR_NAMES AS T1 JOIN CARS_DATA AS T2 ON T1.MakeId = T2.Id WHERE T2.cylinders = 3 ORDER BY T2.horsepower DESC LIMIT 1;","(sqlite3.OperationalError) no such column: Horsepower\n[SQL: SELECT MAX(Horsepower), Maker FROM car_makers WHERE Cylinders = 3;]\n(Background on this error at: https://sqlalche.me/e/20/e3q8)"
2,Which distinctive models are produced by maker with the full name General Motors or weighing more than 3500?,SELECT DISTINCT car_models.Model FROM car_models JOIN car_makers ON car_models.Maker = car_makers.Maker WHERE car_makers.FullName = 'General Motors' OR car_models.Weight > 3500;,SELECT DISTINCT T2.Model FROM CAR_NAMES AS T1 JOIN MODEL_LIST AS T2 ON T1.Model = T2.Model JOIN CAR_MAKERS AS T3 ON T2.Maker = T3.Id JOIN CARS_DATA AS T4 ON T1.MakeId = T4.Id WHERE T3.FullNa...,(sqlite3.OperationalError) no such column: Model\n[SQL: SELECT DISTINCT Model FROM car_makers WHERE FullName = 'General Motors' OR Weight > 3500;]\n(Background on this error at: https://sqlalche.m...
3,"For a volvo model, how many cylinders does the version with least accelerate have?",SELECT c.Cylinders FROM car_makers m JOIN car_models c ON c.Maker = m.Id WHERE m.Maker = 'volvo' ORDER BY c.Acceleration ASC LIMIT 1;,SELECT T1.cylinders FROM CARS_DATA AS T1 JOIN CAR_NAMES AS T2 ON T1.Id = T2.MakeId WHERE T2.Model = 'volvo' ORDER BY T1.accelerate ASC LIMIT 1;,schema_validation: unknown table 'car_models'; unknown table or alias 'c' for column 'Cylinders'; unknown table or alias 'c' for column 'Maker'; unknown table or alias 'c' for column 'Acceleration'
4,What are the different models wthat are lighter than 3500 but were not built by the Ford Motor Company?,SELECT DISTINCT m.Model FROM models m JOIN car_makers cm ON m.Maker = cm.Id WHERE m.Weight < 3500 AND cm.FullName != 'Ford Motor Company';,SELECT DISTINCT T1.model FROM MODEL_LIST AS T1 JOIN CAR_NAMES AS T2 ON T1.Model = T2.Model JOIN CARS_DATA AS T3 ON T2.MakeId = T3.Id JOIN CAR_MAKERS AS T4 ON T1.Maker = T4.Id WHERE T3.weight...,schema_validation: unknown table 'models'; unknown table or alias 'm' for column 'Model'; unknown table or alias 'm' for column 'Maker'; unknown table or alias 'm' for column 'Weight'


In [4]:
# Detailed failed examples inside the main problematic DBs
focus_rows = []
for row in predictions:
    if row['db_id'] in focus_dbs and (row.get('error_message') or not row.get('execution_match')):
        focus_rows.append({
            'db_id': row['db_id'],
            'question': row['question'],
            'predicted_sql': row['predicted_sql'],
            'gold_sql': row['gold_sql'],
            'execution_match': row['execution_match'],
            'error_message': row.get('error_message'),
            'warnings': row.get('warnings', []),
        })

print('Failed / mismatched rows in focus DBs:', len(focus_rows))
for row in focus_rows[:20]:
    print('\n' + '-' * 100)
    pprint(row, width=140)

Failed / mismatched rows in focus DBs: 140

----------------------------------------------------------------------------------------------------
{'db_id': 'car_1',
 'error_message': None,
 'execution_match': False,
 'gold_sql': 'SELECT T1.FullName ,  T1.Id ,  count(*) FROM CAR_MAKERS AS T1 JOIN MODEL_LIST AS T2 ON T1.Id  =  T2.Maker GROUP BY T1.Id;',
 'predicted_sql': 'SELECT FullName, Id, 0 AS NumberOfModels FROM car_makers;',
 'question': 'How many models does each car maker produce? List maker full name, id and the number.',
 'warnings': ['selector: padded reranker selection with vector candidates',
              'query_sketcher: failed to parse JSON response',
              'query_sketcher: structured repair unavailable (Could not parse response content as the length limit was reached - '
              'CompletionUsage(completion_tokens=2032, prompt_tokens=224, total_tokens=2256, '
              'completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, aud

In [5]:
# Clearly wrong predictions outside the main problematic DBs
# These are especially useful before prompt tuning because they reveal global failure modes.
other_failures = []
for row in predictions:
    if row['db_id'] in focus_dbs:
        continue
    if row.get('error_message') or not row.get('execution_match'):
        other_failures.append({
            'db_id': row['db_id'],
            'question': row['question'],
            'predicted_sql': row['predicted_sql'],
            'gold_sql': row['gold_sql'],
            'execution_match': row['execution_match'],
            'error_message': row.get('error_message'),
            'warnings': row.get('warnings', []),
        })

print('Other failed / mismatched rows:', len(other_failures))
for row in other_failures[:20]:
    print('\n' + '-' * 100)
    pprint(row, width=140)

Other failed / mismatched rows: 146

----------------------------------------------------------------------------------------------------
{'db_id': 'concert_singer',
 'error_message': "schema_validation: unknown column 'c.Attendance'",
 'execution_match': False,
 'gold_sql': 'SELECT name ,  capacity FROM stadium ORDER BY average DESC LIMIT 1',
 'predicted_sql': 'SELECT s.Name, s.Capacity FROM stadium s JOIN concert c ON s.Stadium_ID = c.Stadium_ID GROUP BY s.Stadium_ID, s.Name, '
                  's.Capacity ORDER BY AVG(c.Attendance) DESC LIMIT 1;',
 'question': 'What is the name and capacity for the stadium with highest average attendance?',
 'warnings': ['selector: padded reranker selection with vector candidates',
              'generator: using reduced ensemble budget for moderate complexity',
              'execution_filter: (sqlite3.OperationalError) no such column: Attendance\n'
              '[SQL: SELECT Name, Capacity FROM stadium GROUP BY Stadium_ID ORDER BY AVG(Attendance

In [6]:
# Heuristic buckets for fast prompt-tuning ideas
buckets = Counter()
for row in predictions:
    sql = (row.get('predicted_sql') or '').upper()
    warnings = ' | '.join(row.get('warnings', []))
    err = row.get('error_message') or ''

    if 'TRIM(' in sql or 'LOWER(' in sql or 'UPPER(' in sql or 'CAST(' in sql:
        buckets['value_normalization_or_cast'] += 1
    if 'schema_validation:' in warnings or 'schema_validation:' in err:
        buckets['schema_grounding_failures'] += 1
    if 'failed to parse JSON response' in warnings:
        buckets['sketch_parse_failures'] += 1
    if 'repaired malformed sketch' in warnings:
        buckets['sketch_repair_used'] += 1
    if 'skipped judge for safe simple query' in warnings:
        buckets['cheap_path_used'] += 1

buckets

Counter({'cheap_path_used': 402,
         'sketch_parse_failures': 111,
         'schema_grounding_failures': 73,
         'value_normalization_or_cast': 60,
         'sketch_repair_used': 59})

In [7]:
# Helper: inspect one database in detail
def inspect_db(db_id: str, limit: int = 20):
    rows = []
    for row in predictions:
        if row['db_id'] != db_id:
            continue
        if row.get('error_message') or not row.get('execution_match'):
            rows.append(row)
    print(f'{db_id}: {len(rows)} failed/mismatched rows')
    for row in rows[:limit]:
        print('\n' + '-' * 100)
        print('Q:', row['question'])
        print('PRED:', row['predicted_sql'])
        print('GOLD:', row['gold_sql'])
        print('EXEC:', row['execution_match'], 'ERR:', row.get('error_message'))
        print('WARNINGS:', row.get('warnings', []))

# Example:
# inspect_db('car_1')

In [8]:
inspect_db('car_1')

car_1: 59 failed/mismatched rows

----------------------------------------------------------------------------------------------------
Q: How many models does each car maker produce? List maker full name, id and the number.
PRED: SELECT FullName, Id, 0 AS NumberOfModels FROM car_makers;
GOLD: SELECT T1.FullName ,  T1.Id ,  count(*) FROM CAR_MAKERS AS T1 JOIN MODEL_LIST AS T2 ON T1.Id  =  T2.Maker GROUP BY T1.Id;
EXEC: False ERR: None
WARNINGS: ['selector: padded reranker selection with vector candidates', 'query_sketcher: failed to parse JSON response', "query_sketcher: structured repair unavailable (Could not parse response content as the length limit was reached - CompletionUsage(completion_tokens=2032, prompt_tokens=224, total_tokens=2256, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=1965, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0, 